# Analisis Harga Pasaran & Strategi Bisnis Supermarket (Zona USD & EUR)
**Berdasarkan Hasil Co-Creation Data Retail Global (`global_retail_prices_dataset.csv`)**

Notebook ini mencakup 8 tahapan analisis terstruktur sesuai hasil *co-creation*:
1. **Data Cleaning & Referensi Metodologi** (Penanganan *missing values*, *zero-variance*, *schema misalignment*, dan *deduplication* beserta tautan sumber literatur).
2. **Analisis Imbalanced Data** pada distribusi mata uang beserta landasan ilmiah mengapa diperbolehkan memfokuskan analisis pada subset mayoritas (**USD & EUR**).
3. **Analisis Terpisah per Supermarket** yang menggunakan mata uang USD dan EUR.
4. **Eksplorasi Harga Pasaran (*Market Price Benchmarking*)** untuk barang-barang di zona USD dan EUR.
5. **Klasifikasi Frekuensi Pembelian Barang** menjadi barang **Harian/Mingguan (Sering Dibeli)** dan **Bulanan/Tahunan (Jarang Dibeli)**.
6. **Komparasi Supermarket Termurah** di masing-masing zona (USD dan EUR) terhadap barang yang sering dibeli maupun jarang dibeli.
7. **Insight Harga Pasaran Pembukaan Supermarket Baru** di kedua zona.
8. **Rekomendasi Strategi Bisnis & Penetapan Harga (*Retail Pricing Strategy*)** (Strategi *Everyday Low Price* + Event Diskon Awal Bulan vs Strategi *Wholesale/Pantry Anchor*).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
sns.set_theme(style='whitegrid', palette='muted')

# 0. Memuat Dataset Mentah
df_raw = pd.read_csv('global_retail_prices_dataset.csv')
print(f'Ukuran Data Mentah: {df_raw.shape[0]} baris x {df_raw.shape[1]} kolom')
display(df_raw.head())

print('\nInformasi Missing Values pada Data Mentah:')
missing_summary = pd.DataFrame({
    'Missing Count': df_raw.isnull().sum(),
    'Missing Percentage (%)': (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'Unique Values': df_raw.nunique(dropna=False)
})
display(missing_summary)

---
## 1. Pembersihan Data (*Data Cleaning*) & Tautan Sumber Metodologi

Berdasarkan inspeksi pada `df_raw` (2.000 baris x 12 kolom), terdapat **5 kondisi ketidaksempurnaan data** yang harus dibersihkan terlebih dahulu. Berikut adalah rincian penanganan untuk setiap kondisi beserta referensi metodologinya:

### Kondisi 1.1: Kolom 100% Kosong (`brand`, `country_code`) & Kolom *Zero-Variance* (`price_is_discounted`)
* **Masalah:** Kolom `brand` dan `country_code` memiliki 2.000 nilai `NaN` (100% kosong). Selain itu, kolom `price_is_discounted` hanya memiliki satu nilai tunggal yaitu `False` pada seluruh 2.000 baris (variansi = 0), sehingga ketiganya tidak memberikan informasi pembeda (*zero information gain*).
* **Penanganan:** Menghapus ketiga kolom tersebut menggunakan `.drop(columns=[...])`.
* **Sumber Referensi:**
  * [Pandas Documentation — Working with Missing Data (`dropna` & dropping empty columns)](https://pandas.pydata.org/docs/user_guide/missing_data.html)
  * [Scikit-Learn Documentation — Feature Selection: Removing Features with Low/Zero Variance (`VarianceThreshold`)](https://scikit-learn.org/stable/modules/feature_selection.html#removing-features-with-low-variance)

### Kondisi 1.2: Pergeseran Label Kolom (*Schema Misalignment*) & Inkonsistensi Penulisan Nama Supermarket
* **Masalah:** Pada data mentah, kolom `store_name` berisi *username* pencatat/kontributor data (seperti `poslovitch`, `doby`, `jbieber`), sedangkan nama supermarket tempat barang dijual justru tercatat di kolom `city` (seperti `Dumbo Market`, `E. Leclerc`, `Carrefour Market`). Selain itu, terdapat variasi penulisan nama jaringan ritel yang sama (misalnya `REWE` dengan huruf kapital semua vs `Rewe`, atau cabang spesifik `E.Leclerc Levallois` vs `E. Leclerc`).
* **Penanganan:** Mengubah nama kolom `city` menjadi `supermarket_name` dan `store_name` menjadi `contributor_id`, serta melakukan normalisasi string (`.str.strip()` dan `.replace()`) untuk menyatukan penulisan nama supermarket.
* **Sumber Referensi:**
  * [Pandas Documentation — Working with Text Data & String Normalization](https://pandas.pydata.org/docs/user_guide/text.html)
  * [OpenRefine / Data Carpentry — Data Cleaning & Text Faceting/Clustering Best Practices](https://datacarpentry.org/OpenRefine-ecology-lesson/01-working-with-openrefine/index.html)

### Kondisi 1.3: *Missing Values* Parsial pada `product_name` (461 Baris Kosong)
* **Masalah:** Terdapat 461 baris yang tidak memiliki `product_name`. Jika langsung dihapus semuanya, kita akan kehilangan banyak data harga yang sebenarnya masih bisa diidentifikasi melalui barcode (`product_code`) atau label kategori (`category`).
* **Penanganan Berjenjang (*Hierarchical Imputation & Listwise Deletion*):**
  1. **Deterministic Barcode Imputation:** Memetakan `product_code` (EAN/barcode) yang sama dari baris lain yang memiliki nama produk lengkap. Langkah ini berhasil memulihkan **11 baris**.
  2. **Category-Based Imputation:** Mengisi `product_name` yang masih kosong menggunakan nilai dari kolom `category` setelah dibersihkan dari awalan `en:` (misalnya `en:yogurts` -> `Yogurts`). Langkah ini berhasil memulihkan **115 baris**.
  3. **Listwise Deletion untuk Unidentifiable Records:** Sebanyak **335 baris** sisanya tidak memiliki `product_name` maupun `category` sama sekali (hanya angka harga tanpa identitas produk). Baris-baris ini dihapus karena tidak dapat diklasifikasikan ke dalam jenis barang harian maupun bulanan.
* **Sumber Referensi:**
  * [Stef van Buuren — Flexible Imputation of Missing Data (Chapter 1: Deductive Imputation & Listwise Deletion)](https://stefvanbuuren.name/fimd/sec-simplesolutions.html)
  * [Pandas Documentation — Filling Missing Values (`fillna` & `map`)](https://pandas.pydata.org/docs/reference/api/pandas.Series.fillna.html)

### Kondisi 1.4: Duplikasi Pencatatan Transaksi Identik (*Exact Duplicate Records*)
* **Masalah:** Meskipun kolom `price_record_id` bersifat unik untuk setiap baris, terdapat **26 baris duplikat** yang memiliki nilai `product_code`, `product_name`, `price`, `currency`, `supermarket_name`, dan `date_recorded` yang sama persis (tercatat dua kali di toko dan hari yang sama).
* **Penanganan:** Menghapus baris duplikat menggunakan `.drop_duplicates()` pada subset atribut transaksi utama agar tidak mendistorsi perhitungan rata-rata dan median harga supermarket.
* **Sumber Referensi:**
  * [Pandas Documentation — Duplicate Labels (`drop_duplicates`)](https://pandas.pydata.org/docs/user_guide/duplicates.html)

In [2]:
# 1.1 Menghapus kolom 100% kosong dan zero-variance
df_clean = df_raw.drop(columns=['brand', 'country_code', 'price_is_discounted']).copy()

# 1.2 Memperbaiki nama kolom & menstandardisasi nama supermarket
df_clean = df_clean.rename(columns={'city': 'supermarket_name', 'store_name': 'contributor_id'})
df_clean['supermarket_name'] = df_clean['supermarket_name'].str.strip().replace({
    'REWE': 'Rewe',
    'E.Leclerc Levallois': 'E. Leclerc',
    'Intermarché Super': 'Intermarché',
    'U Express Lyon Université': 'U Express'
})

# 1.3 Imputasi product_name berjenjang
initial_missing_name = df_clean['product_name'].isnull().sum()

# Tahap A: Imputasi menggunakan product_code (barcode)
code_to_name = (
    df_clean.dropna(subset=['product_code', 'product_name'])
    .groupby('product_code')['product_name']
    .first()
    .to_dict()
)
df_clean['product_name'] = df_clean['product_name'].fillna(df_clean['product_code'].map(code_to_name))
after_barcode_missing = df_clean['product_name'].isnull().sum()

# Tahap B: Imputasi menggunakan kolom category (membersihkan awalan 'en:')
df_clean['category_clean'] = (
    df_clean['category']
    .str.replace('en:', '', regex=False)
    .str.replace('-', ' ', regex=False)
    .str.title()
)
df_clean['product_name'] = df_clean['product_name'].fillna(df_clean['category_clean'])
after_category_missing = df_clean['product_name'].isnull().sum()

# Tahap C: Menghapus baris yang tetap tidak memiliki identitas produk (NaN pada product_name dan category)
df_clean = df_clean.dropna(subset=['product_name']).copy()
rows_after_dropna = len(df_clean)

# 1.4 Menghapus duplikat pencatatan transaksi identik
df_clean = df_clean.drop_duplicates(
    subset=['product_code', 'product_name', 'price', 'currency', 'supermarket_name', 'date_recorded']
).copy()
rows_final_clean = len(df_clean)

print('=== RINGKASAN AUDIT PEMBERSIHAN DATA ===')
print(f'1. Baris awal dataset mentah                      : {len(df_raw)} baris')
print(f'2. Missing product_name awal                      : {initial_missing_name} baris')
print(f'   - Dipulihkan lewat Barcode (product_code)      : {initial_missing_name - after_barcode_missing} baris')
print(f'   - Dipulihkan lewat Kategori (category)         : {after_barcode_missing - after_category_missing} baris')
print(f'   - Dihapus karena tanpa identitas sama sekali   : {after_category_missing} baris')
print(f'3. Baris duplikat transaksi identik yang dihapus  : {rows_after_dropna - rows_final_clean} baris')
print(f'4. Total baris data bersih siap analisis          : {rows_final_clean} baris')

---
## 2. Evaluasi *Imbalanced Data* pada Mata Uang & Landasan Ilmiah Fokus ke USD dan EUR

Sebelum membagi analisis ke zona mata uang tertentu, mari kita buktikan terlebih dahulu bahwa distribusi data berdasarkan mata uang (`currency`) mengalami **ketimpangan jumlah sampel (*data imbalance*) yang sangat tajam**.

### Apakah boleh jika datanya *imbalanced* kita memfokuskan analisis hanya pada kelompok mayoritas (USD dan EUR)?
**Jawabannya: Sangat boleh dan secara statistik justru direkomendasikan** untuk tujuan *market price benchmarking* dan perumusan strategi bisnis. Berikut adalah 3 alasan metodologis beserta rujukan ilmiahnya:

1. **Menghindari *Small Sample Bias* & *High Variance Estimator*:**
   Dalam statistika inferensial dan *Exploratory Data Analysis (EDA)*, menghitung rata-rata atau median harga pasar dari kelompok mata uang yang hanya memiliki 1 hingga 17 sampel (seperti THB = 1 baris, PLN = 13 baris, SGD = 16 baris, KZT = 17 baris) akan menghasilkan *standard error* yang sangat lebar ($SE = \frac{s}{\sqrt{n}}$). Sampel mikro tersebut tidak merepresentasikan harga pasar sesungguhnya di negara terkait (misalnya 1 transaksi di Thailand tidak bisa dijadikan patokan harga supermarket nasional).
2. **Prinsip *Subpopulation / Domain Analysis* & *Minimum Sample Size Threshold*:**
   Ketika sebuah dataset global dikumpulkan secara *crowdsourced* dan menghasilkan distribusi *long-tail* (di mana **EUR = 825 baris [50,34%]** dan **USD = 405 baris [24,71%]** menguasai **75,05%** dari seluruh data bersih), membatasi ruang lingkup inferensi (*scope restriction*) pada domain/sub-populasi yang memenuhi syarat kecukupan ukuran sampel ($n \ge 30$ hingga ratusan observasi per zona) adalah prosedur standar untuk menjaga validitas internal.
3. **Mencegah *Cross-Currency Distortion*:**
   Setiap zona mata uang memiliki daya beli (*Purchasing Power Parity*), struktur pajak (PPN/VAT), dan skala nominal yang berbeda, sehingga analisis komparasi harga supermarket wajib dipisahkan per zona mata uang yang memiliki kepadatan sampel memadai.

### Tautan Sumber Literatur Terkait Penanganan *Imbalanced Data* & *Subpopulation Filtering*:
* **NIST/SEMATECH e-Handbook of Statistical Methods (Exploratory Data Analysis & Sample Size Requirements):** [https://www.itl.nist.gov/div898/handbook/eda/section1/eda11.htm](https://www.itl.nist.gov/div898/handbook/eda/section1/eda11.htm)
* **Google Developers Machine Learning Crash Course — Imbalanced Datasets & Distribution Bias:** [https://developers.google.com/machine-learning/crash-course/overfitting/imbalanced-datasets](https://developers.google.com/machine-learning/crash-course/overfitting/imbalanced-datasets)
* **PennState Eberly College of Science (STAT 504) — Stratified Analysis & Sparse Data Filtering:** [https://online.stat.psu.edu/stat504/](https://online.stat.psu.edu/stat504/)

In [3]:
# 2.1 Tabel & Visualisasi Ketimpangan Distribusi Mata Uang (Imbalanced Data)
curr_table = df_clean['currency'].value_counts().reset_index()
curr_table.columns = ['Currency', 'Clean Row Count']
curr_table['Share (%)'] = (curr_table['Clean Row Count'] / len(df_clean) * 100).round(2)
curr_table['Cumulative Share (%)'] = curr_table['Share (%)'].cumsum().round(2)
curr_table['Status Analisis'] = np.where(
    curr_table['Currency'].isin(['EUR', 'USD']),
    'Fokus Analisis Utama (Sampel Memadai)',
    'Dieliminasi (Sampel Kecil / Minoritas)'
)
display(curr_table)

# Visualisasi Imbalance Mata Uang
plt.figure(figsize=(10, 5))
colors = ['#1f77b4' if c in ['EUR', 'USD'] else '#c7c7c7' for c in curr_table['Currency']]
ax = sns.barplot(data=curr_table, x='Currency', y='Clean Row Count', palette=colors)
plt.title('Distribusi Data Bersih Berdasarkan Mata Uang (Imbalanced Dataset: EUR & USD = 75.05%)', fontsize=12, fontweight='bold')
plt.xlabel('Mata Uang (Currency)')
plt.ylabel('Jumlah Produk Tercatat (Baris)')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

# Memisahkan dataset menjadi Zona USD dan Zona EUR
df_usd = df_clean[df_clean['currency'] == 'USD'].copy()
df_eur = df_clean[df_clean['currency'] == 'EUR'].copy()
print(f'Subset Fokus Siap Analisis -> Zona EUR: {len(df_eur)} produk | Zona USD: {len(df_usd)} produk')

---
## 3 & 4. Analisis Terpisah per Supermarket & Gambaran Harga Pasaran (*Market Price*) di Zona USD dan EUR

Pada tahap ini kita melakukan analisis terpisah untuk mengetahui:
1. Barang-barang apa saja yang dijual di masing-masing supermarket zona **USD** dan **EUR**.
2. Berapa **harga pasaran wajar (*Median* dan *Interquartile Range* $Q_1 - Q_3$)** serta rata-rata (*Mean*) di masing-masing zona dan supermarket.

> **Catatan Statistik:** Karena harga produk ritel sering memiliki kemencengan positif (*right-skewed* akibat produk kemasan besar atau barang premium seperti *Manuka Honey* $43,99 atau *Olive Oil 2L* €25,87), kita menggunakan **Median ($Q_2$)** bersama **Mean** sebagai indikator utama harga pasaran wajar.

In [4]:
# 3.1 Ringkasan Harga Pasaran Keseluruhan Zona USD vs Zona EUR
market_overview = pd.DataFrame({
    'Zona Mata Uang': ['USD ($)', 'EUR (€)'],
    'Jumlah Produk': [len(df_usd), len(df_eur)],
    'Jumlah Supermarket': [df_usd['supermarket_name'].nunique(), df_eur['supermarket_name'].nunique()],
    'Mean Price': [df_usd['price'].mean().round(2), df_eur['price'].mean().round(2)],
    'Median Price (Harga Pasaran Wajar)': [df_usd['price'].median().round(2), df_eur['price'].median().round(2)],
    'Q1 (Batas Bawah Pasar)': [df_usd['price'].quantile(0.25).round(2), df_eur['price'].quantile(0.25).round(2)],
    'Q3 (Batas Atas Pasar)': [df_usd['price'].quantile(0.75).round(2), df_eur['price'].quantile(0.75).round(2)],
    'Min Price': [df_usd['price'].min().round(2), df_eur['price'].min().round(2)],
    'Max Price': [df_usd['price'].max().round(2), df_eur['price'].max().round(2)],
    'Std Dev': [df_usd['price'].std().round(2), df_eur['price'].std().round(2)]
})
print('=== HARGA PASARAN KESELURUHAN ZONA USD vs EUR ===')
display(market_overview)

# 3.2 Tabel Profil Harga & Contoh Barang per Supermarket di Zona USD
usd_store_summary = df_usd.groupby('supermarket_name').agg(
    jumlah_barang=('price', 'count'),
    mean_price=('price', lambda x: round(x.mean(), 2)),
    median_price=('price', lambda x: round(x.median(), 2)),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    contoh_barang_dijual=('product_name', lambda x: ', '.join(x.head(4)))
).sort_values(by='jumlah_barang', ascending=False)

print('\n=== PROFIL SUPERMARKET & BARANG YANG DIJUAL DI ZONA USD ($) ===')
display(usd_store_summary)

# 3.3 Tabel Profil Harga & Contoh Barang per Supermarket di Zona EUR (Minimal 4 Sampel Produk)
eur_store_summary = df_eur.groupby('supermarket_name').agg(
    jumlah_barang=('price', 'count'),
    mean_price=('price', lambda x: round(x.mean(), 2)),
    median_price=('price', lambda x: round(x.median(), 2)),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    contoh_barang_dijual=('product_name', lambda x: ', '.join(x.head(4)))
).sort_values(by='jumlah_barang', ascending=False)

print('\n=== PROFIL SUPERMARKET & BARANG YANG DIJUAL DI ZONA EUR (€) (Top 15 Supermarket) ===')
display(eur_store_summary.head(15))

---
## 5 & 6. Pembagian Kategori Frekuensi Pembelian (Sering vs Jarang Dibeli) & Evaluasi Supermarket Termurah

Selanjutnya kita membagi seluruh produk di zona USD dan EUR ke dalam 2 kelompok berdasarkan siklus belanja konsumen:
1. **Harian/Mingguan (Sering Dibeli / *Fast-Moving Daily-Weekly Items*):**
   * Mencakup kebutuhan segar dan konsumsi cepat habis seperti roti, susu, yogurt, keju, buah & sayur segar, daging/sosis/produk nabati siap masak, makanan beku/siap saji, es krim *pint*, camilan/keripik, cokelat batangan, dan minuman ringan.
2. **Bulanan/Tahunan (Jarang Dibeli / *Slow-Moving Pantry & Stock Items*):**
   * Mencakup barang stok dapur dan kebutuhan rumah tangga berumur simpan panjang seperti minyak zaitun/minyak goreng (*olive oil, walnut oil, safflower oil*), madu (*honey/Manuka*), sirup *maple*, garam/lada/bumbu rempah kering, saus botol/mayones/mustard, selai/olesan coklat (*Nutella*), kaldu kemasan (*broth/stock*), bahan kue/tepung, kopi biji 1 kg/kapsul, teh celup, popok bayi (*diapers/couches Pampers*), minuman beralkohol botol besar (*Champagne, Liqueur*), dan alat rumah tangga (*bath scale, LED*).

In [5]:
# 5.1 Kamus Kata Kunci untuk Barang Bulanan/Tahunan (Jarang Dibeli)
rare_keywords = [
    # Minyak, Cuka, Madu, Sirup
    'OIL', 'HUILE', 'OLIVE', 'EVOO', 'TRUFFL', 'TRFL', 'VINEGAR', 'VINAIGRE',
    'HONEY', 'HNY', 'MIEL', 'MANUKA', 'SYRUP', 'SIROP', 'MAPLE', 'MOLASSES', 'AGAVE',
    # Bumbu, Rempah, Garam, Lada, Bahan Kue
    'SALT', 'SEL ', 'PEPPER', 'POIVRE', 'SPICE', 'SEASONING', 'SSNG', 'GARLIC PWDR', 'POWDER',
    'CINNAMON', 'CUMIN', 'PAPRIKA', 'OREGANO', 'TURMERIC', 'CURRY', 'HERB', 'VANILLA BEAN',
    'EXTRACT', 'FLOUR', 'FARINE', 'SUGAR', 'SUCRE', 'BAKING', 'CAKE MIX', 'FROSTING', 'YEAST', 'LEVURE',
    # Saus Botol, Kondimen, Selai, Olesan, Kaldu, Makanan Kaleng/Botol Kaca
    'SAUCE', 'SCE', 'MAYONNAISE', 'MAYO', 'MUSTARD', 'MOUTARDE', 'KETCHUP', 'PESTO', 'ALFREDO',
    'MARINARA', 'SALSA', 'HOT SAUCE', 'SOY SAUCE', 'TAMARI', 'TERIYAKI', 'NUOC MAM', 'BBQ',
    'JAM', 'JELLY', 'CONFITURE', 'PRESERVE', 'MARMALADE', 'SPREAD', 'NUTELLA', 'NUTELA',
    'PEANUT BUTTER', 'CASHEW BUTTER', 'ALMOND BUTTER', 'TAHINI', 'PATE DE SESAME',
    'BROTH', 'STOCK', 'BOUILLON', 'CANNED', 'CONSERVE', 'BOCAL', 'PICKLE', 'CORNICHON', 'CAPERS', 'OLIVES',
    # Popok Bayi, Rumah Tangga, Minuman Keras Botol, Kopi/Teh Stok
    'COUCHES', 'PAMPERS', 'BABYDRY', 'BABY DRY', 'DIAPER', 'BATH SCALE', 'MONSTER LED',
    'CHAMPAGNE', 'LIQUEUR', 'JAGERMEISTER', 'WHISKY', 'VODKA', 'GIN ', 'RUM ', 'RHUM', 'PICON', 'GET 31', 'VIN ', 'WINE',
    '1KG GRAINS', 'COFFEE BEANS', 'GROUND COFFEE', 'STARBUCK', 'CAPS', 'CAPSULES', 'NESCAF', 'CAFE',
    'TEA ', 'INFUS', 'TISANE', 'TALLOW', 'GHEE', 'LARD'
]

def classify_purchase_frequency(row):
    combined_text = f"{str(row['product_name']).upper()} {str(row['category']).upper()}"
    for kw in rare_keywords:
        if kw in combined_text:
            return 'Bulanan/Tahunan (Jarang Dibeli)'
    return 'Harian/Mingguan (Sering Dibeli)'

df_usd['freq_category'] = df_usd.apply(classify_purchase_frequency, axis=1)
df_eur['freq_category'] = df_eur.apply(classify_purchase_frequency, axis=1)

print('=== HARGA PASARAN ZONA USD ($) BERDASARKAN FREKUENSI PEMBELIAN ===')
display(df_usd.groupby('freq_category')['price'].agg(['count', 'mean', 'median', 'min', 'max', 'std']).round(2))

print('\n=== HARGA PASARAN ZONA EUR (€) BERDASARKAN FREKUENSI PEMBELIAN ===')
display(df_eur.groupby('freq_category')['price'].agg(['count', 'mean', 'median', 'min', 'max', 'std']).round(2))

In [6]:
# 6.1 Komparasi Supermarket Termurah di Zona USD untuk Barang Sering vs Jarang Dibeli
usd_comp = df_usd.pivot_table(
    index='supermarket_name',
    columns='freq_category',
    values='price',
    aggfunc=['count', 'mean', 'median']
).round(2)
print('=== PERBANDINGAN HARGA SUPERMARKET ZONA USD ($): SERING vs JARANG DIBELI ===')
display(usd_comp)

# 6.2 Komparasi Supermarket Termurah di Zona EUR (Supermarket dengan >= 8 sampel produk)
top_eur_stores = df_eur['supermarket_name'].value_counts()[lambda x: x >= 8].index
eur_comp = df_eur[df_eur['supermarket_name'].isin(top_eur_stores)].pivot_table(
    index='supermarket_name',
    columns='freq_category',
    values='price',
    aggfunc=['count', 'mean', 'median']
).round(2)
print('\n=== PERBANDINGAN HARGA SUPERMARKET UTAMA ZONA EUR (€): SERING vs JARANG DIBELI ===')
display(eur_comp)

In [7]:
# 6.3 Visualisasi Perbandingan Harga Median Supermarket (Barang Sering vs Jarang Dibeli)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot USD
usd_plot_df = df_usd.groupby(['supermarket_name', 'freq_category'])['price'].median().reset_index()
sns.barplot(data=usd_plot_df, x='supermarket_name', y='price', hue='freq_category', ax=axes[0])
axes[0].set_title('Zona USD ($): Median Harga per Supermarket', fontweight='bold')
axes[0].set_xlabel('Supermarket (USD)')
axes[0].set_ylabel('Median Harga ($)')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Frekuensi Beli')

# Plot EUR (Top Stores)
eur_plot_df = df_eur[df_eur['supermarket_name'].isin(top_eur_stores)].groupby(['supermarket_name', 'freq_category'])['price'].median().reset_index()
sns.barplot(data=eur_plot_df, x='supermarket_name', y='price', hue='freq_category', ax=axes[1])
axes[1].set_title('Zona EUR (€): Median Harga per Supermarket Utama', fontweight='bold')
axes[1].set_xlabel('Supermarket (EUR)')
axes[1].set_ylabel('Median Harga (€)')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(title='Frekuensi Beli')

plt.tight_layout()
plt.show()

---
## 7 & 8. Insight Harga Pasaran & Strategi Bisnis Membuka Supermarket di Zona USD dan EUR

### A. Ringkasan Supermarket Termurah di Setiap Zona
1. **Zona USD ($):**
   * **Barang Sering Dibeli (Harian/Mingguan):** **Smart & Final extra!** (Mean **$4,12** | Median **$3,99**) dan **ShopRite** (Median **$3,99** | Mean **$6,34**) adalah yang paling murah, disusul **Safeway** (Mean **$5,44** | Median **$4,99**). Sebaliknya, **Dumbo Market** (Mean **$10,81** | Median **$8,24**) dan **Whole Foods Market** (Mean **$9,81** | Median **$6,69**) memiliki harga tertinggi karena berfokus pada segmen *gourmet* dan organik.
   * **Barang Jarang Dibeli (Bulanan/Tahunan):** Di antara supermarket dengan koleksi stok bulanan yang memadai, **Smart & Final extra!** (Mean **$5,77**) dan **Dumbo Market** (Median **$9,99** | Mean **$11,15**) jauh lebih terjangkau dibandingkan **Whole Foods Market** (Mean **$16,67** | Median **$12,64**).
2. **Zona EUR (€):**
   * **Barang Sering Dibeli (Harian/Mingguan):** Untuk format *hard-discounter*, **Penny** (Mean **€1,11** | Median **€1,27**), **Hofer** (Mean **€1,26** | Median **€1,37**), dan **Norma** (Mean **€1,74** | Median **€1,99**) memimpin harga termurah. Untuk kategori *full-service supermarket/hypermarket*, **E. Leclerc** adalah yang termurah (Median **€2,04** | Mean **€2,61**), mengungguli **Colruyt** (Median **€2,19**) dan **Carrefour Market** (Median **€2,80**).
   * **Barang Jarang Dibeli (Bulanan/Tahunan):** **E. Leclerc** kembali menjadi yang termurah (Mean **€2,77** | Median **€2,13**), diikuti oleh **Carrefour Market** (Median **€2,29** | Mean **€3,74**) dan **Colruyt** (Median **€2,37** | Mean **€4,70**).

---
### B. Patokan Harga Pasaran (*Price Benchmark*) Jika Membuka Supermarket Baru
* **Jika Membuka Supermarket di Zona USD ($):**
  * **Patokan Harga Pasar Saat Ini:** Barang Harian/Mingguan memiliki median pasar **$6,99** (rata-rata **$9,56**), sedangkan Barang Bulanan/Tahunan memiliki median pasar **$10,49** (rata-rata **$12,17**).
  * **Target Harga Kompetitif:** Pasang harga barang harian di rentang **$3,99 – $5,49** dan barang bulanan di rentang **$8,99 – $10,49**.
* **Jika Membuka Supermarket di Zona EUR (€):**
  * **Patokan Harga Pasar Saat Ini:** Barang Harian/Mingguan memiliki median pasar **€2,48** (rata-rata **€2,93**), sedangkan Barang Bulanan/Tahunan memiliki median pasar **€2,38** (rata-rata **€3,88**).
  * **Target Harga Kompetitif:** Pasang harga barang harian di rentang **€1,85 – €2,20** (menyaingi E. Leclerc & Colruyt) dan barang bulanan di rentang **€2,15 – €3,50**.

---
### C. Dua Skenario Strategi Penetapan Harga (*Retail Pricing Strategy*)

#### Skenario 1: Condong Murah pada Barang Sering Dibeli (*EDLP Traffic Builder* + Promo Gajian/Event untuk Barang Bulanan)
* **Mekanisme Harga:** Menetapkan harga sangat kompetitif pada barang harian/mingguan (roti, susu, telur, sayur, camilan di kisaran **€1,90 – €2,15** atau **$4,25 – $5,25**) untuk membangun kebiasaan belanja mingguan pelanggan (*high foot-traffic*). Sementara itu, barang bulanan/tahunan (minyak zaitun, madu, kopi 1 kg, popok bayi, bumbu) ditetapkan pada margin normal cenderung tinggi.
* **Strategi Promosi Barang Bulanan:**
  1. **Payday Big Pack Promo (Setiap Awal Bulan Tanggal 25–5):** Memberikan diskon 15%–25% atau *Bundle Buy 2 Save More* khusus untuk barang bulanan (minyak, kopi, popok, saus) bertepatan dengan siklus gaji bulanan pelanggan.
  2. **Seasonal & Holiday Pantry Event:** Mengadakan promo tematik menjelang hari besar (misalnya *Baking & Cooking Fest* untuk tepung, sirup, mentega, dan bumbu rempah).
  3. **Minimum Basket Voucher:** Setiap belanja barang harian senilai $30 / €20, pelanggan mendapat kupon potongan harga khusus untuk menebus 1 barang stok bulanan.

#### Skenario 2 (Sebaliknya): Condong Murah pada Barang Jarang Dibeli (*Pantry/Wholesale Anchor*) + Strategi Nilai Tambah pada Barang Harian
* **Mekanisme Harga:** Mencerminkan pola harga yang terlihat di **Monoprix** (di mana rata-rata barang bulanan mereka **€3,33** justru lebih rendah daripada barang harian/siap saji mereka **€4,31**). Supermarket mematok harga sangat murah pada barang stok bulanan (kopi, minyak, bumbu, kebutuhan rumah tangga) agar pelanggan menjadikan toko kita destinasi utama belanja bulanan.
* **Ide Strategi Bisnis untuk Mengoptimalkan Penjualan Barang Harian yang Lebih Mahal:**
  1. **Strategi *Ready-to-Eat & Convenience Bundle* (Meal Deal Harian):** Karena harga satuan barang harian lebih tinggi, kemas dalam paket praktis siap santap (misalnya *Lunch Deal*: 1 Sandwich/Salad + 1 Minuman + 1 Yogurt/Dessert dengan harga paket hemat) yang menarik pekerja dan mahasiswa.
  2. **Dynamic Markdown / *Evening Flash Sale* (Diskon Jam Malam):** Untuk barang harian segar (roti, salad, produk susu, makanan siap saji) yang mendekati batas simpan harian, berikan diskon otomatis 30%–50% setelah pukul 19.00. Strategi ini menjaga *zero food waste* sekaligus menarik pembeli sensitif harga tanpa harus menurunkan harga reguler siang hari.
  3. **Loyalty Tier / Membership Pantry Club:** Pelanggan yang berlangganan *member* mendapatkan akses harga grosir pada barang bulanan, sementara poin *cashback* dari belanja bulanan tersebut hanya bisa ditukarkan untuk membeli produk segar harian di minggu berikutnya (*retention loop*).